# Hospitalization Risk Analysis

This notebook analyzes hospitalization outcomes in the CDC COVID-19 Case Surveillance dataset.

The goal is to measure hospitalization rates across demographic groups, starting with age groups.

Hospitalization rate is defined as:

hospitalized cases / cases with known hospitalization status

Records with unknown or missing hospitalization values are excluded from the denominator.

## Data Preparation

The cleaned dataset produced in `01_data_acquisition_cleaning_q1.ipynb` is loaded.

The following filters are applied:

• Keep only records with known hospitalization status ("Yes" or "No")  
• Remove records with missing or unknown age groups  
• Standardize age group labels by removing the "Years" suffix

These steps ensure hospitalization rates are calculated only on valid observations.

## Hospitalization Rate by Age Group

Hospitalization rates are calculated by grouping records by age group and computing:

hospitalization rate = hospitalized cases / known cases

Where:

• hospitalized cases = records where `hosp_yn = "Yes"`  
• known cases = records where `hosp_yn` is either "Yes" or "No"

## Data Limitation

The dataset used in this project is a 1 million record sample retrieved from the CDC Socrata API.

Because the API returns records in database order rather than randomly, the sample does not include all CDC age categories.

As a result, some age groups (such as 0–9, 60–69, 70–79, and 80+) are not present in this subset.

The analysis therefore reflects hospitalization rates only for the age groups available in the sample.

## Interpretation

Hospitalization rates increase significantly among older age groups in the dataset.

The 40–49 and 50–59 age groups show noticeably higher hospitalization rates compared to younger populations.

This pattern aligns with known epidemiological trends where older populations experience more severe COVID-19 outcomes.

In [ ]:
import pandas as pd

df_clean = pd.read_parquet("../data/processed/cdc_clean_1M.parquet")

df_clean.shape
df_clean.head()

In [ ]:
df_clean["age_group"].value_counts().head(15)

In [ ]:
#Remove invalid hospitalization values
df_known_hosp = df_clean[df_clean["hosp_yn"].isin(["Yes", "No"])]

df_known_hosp["age_group"] = (
    df_known_hosp["age_group"]
    .str.replace(' Years', '', case=False)
    .str.replace(" ", "", regex=False)
)
df_known_hosp = df_known_hosp[
    ~df_known_hosp["age_group"].isin(["Unknown", "Missing"])
]

df_known_hosp.head(5)

In [ ]:
age_order = [
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79",
]

known_cases = (
    df_known_hosp.
    groupby("age_group"). 
    size()
)         

hospitalized = (
    df_known_hosp[df_known_hosp['hosp_yn'] == 'Yes'].
    groupby('age_group').
    size()
)

In [ ]:
hosp_summary = pd.DataFrame({
    "hospitalized": hospitalized,
    "known_cases": known_cases
})
hosp_summary["hospitalization_rate"] = (
    (hosp_summary["hospitalized"]/
    hosp_summary["known_cases"]) * 100
)
hosp_summary = hosp_summary.reindex(age_order)
hosp_summary = hosp_summary.fillna(0)
hosp_summary["hospitalization_rate"] = hosp_summary["hospitalization_rate"].round(2)

In [ ]:
hosp_summary

In [ ]:
hosp_summary.to_csv("../outputs/hospitalization_by_age.csv")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plot_df = hosp_summary[hosp_summary['known_cases'] > 0]
plt.bar(
    plot_df.index,
    plot_df["hospitalization_rate"]
)

plt.xlabel("Age Group")
plt.ylabel("Hospitilization Rate (%)")
plt.title("COVID-19 Hospitilization Rate by Age Group")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig("../outputs/hospitalization_by_age.png")
plt.show()